# 🍏 Health & Fitness Agent with Bing Grounding 🍎

Welcome to our **Health & Fitness Agent with Bing Grounding** tutorial! In this notebook, we'll demonstrate how to:

1. **Initialize** a project using Azure AI Foundry.
2. **Create an Agent** with the **BingGroundingTool** for web search.
3. **Ask real-world questions** about health and fitness.
4. **Retrieve and display** answers, including Bing query URLs and disclaimers.

### ⚠️ Important Model Support Note ⚠️
> Bing grounding is currently only supported in certain Azure OpenAI models (e.g. `gpt-4o-0513`).
> 
> Make sure you specify a supported model and set the `"x-ms-enable-preview": "true"` header.

## Prerequisites
- Complete Agent basics notebook - [1-basics.ipynb](1-basics.ipynb)
- Grounding with Bing connection in Azure AI Foundry, which has to be provisioned from Azure portal.

## Let's Explore Grounding with Bing!
We'll integrate **Grounding with Bing** search results into our agent so it can gather extra context from the web. We'll store and display the Bing search query link for transparency. 🎉

<br/>

<img src="./seq-diagrams/4-bing-grounding.png" width="30%"/>

## 1. Initial Setup
We'll load environment variables from `.env` and initialize our **AIProjectClient** to manage agents.

In [ ]:
import os
import requests
from pathlib import Path
from urllib.parse import urlparse
from dotenv import load_dotenv
from azure.identity import AzureCliCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import BingGroundingTool
from dataclasses import dataclass
import uuid

notebook_path = Path().absolute()
load_dotenv(notebook_path.parent.parent / '.env')
credential = AzureCliCredential()

_url = os.getenv("PROJECT_ENDPOINT")
_parsed = urlparse(_url)
base_endpoint = f"{_parsed.scheme}://{_parsed.netloc}"
path_parts = [p for p in _parsed.path.split("/") if p]
project_name = path_parts[-1] if path_parts else ""
hub_name = _parsed.netloc.split(".")[0]

print("Auto-detecting subscription ID and resource group...")
try:
    mgmt_token = credential.get_token("https://management.azure.com/.default").token
    headers = {"Authorization": f"Bearer {mgmt_token}"}
    subs = requests.get("https://management.azure.com/subscriptions?api-version=2020-01-01", headers=headers, timeout=15).json().get("value", [])
    subscription_id = None
    resource_group = None
    for sub in subs:
        sub_id = sub["subscriptionId"]
        resources = requests.get(f"https://management.azure.com/subscriptions/{sub_id}/resources?$filter=name eq '{hub_name}' and resourceType eq 'Microsoft.CognitiveServices/accounts'&api-version=2021-04-01", headers=headers, timeout=15).json().get("value", [])
        if resources:
            resource_group = resources[0]["id"].split("/")[4]
            subscription_id = sub_id
            break
    if not (subscription_id and resource_group):
        raise RuntimeError(f"Hub '{hub_name}' not found in any subscription.")
    print(f"✓ Subscription ID:  {subscription_id[:8]}...")
    print(f"✓ Resource group:   {resource_group}")
except Exception as e:
    raise EnvironmentError(f"Failed to auto-detect subscription and resource group: {e}") from e

try:
    project_client = AIProjectClient(endpoint=base_endpoint, subscription_id=subscription_id, resource_group_name=resource_group, project_name=project_name, credential=credential)
    print("✅ Successfully initialized AIProjectClient")
except Exception as e:
    print(f"❌ Error initializing project client: {e}")

@dataclass
class LocalBingAgent:
    """Mock agent for fallback when Bing connection unavailable"""
    id: str
    name: str
    instructions: str

def get_foundry_connection(conn_name):
    """Get Bing connection from Foundry"""
    token = credential.get_token("https://management.azure.com/.default").token
    headers = {"Authorization": f"Bearer {token}"}
    for api_ver in ["2025-04-01-preview", "2024-12-01-preview", "2024-10-01-preview"]:
        url = f"https://management.azure.com/subscriptions/{subscription_id}/resourceGroups/{resource_group}/providers/Microsoft.CognitiveServices/accounts/{hub_name}/projects/{project_name}/connections/{conn_name}?api-version={api_ver}&listConnectionSecretsWithAccessKeys=true"
        resp = requests.get(url, headers=headers, timeout=15)
        if resp.status_code == 200:
            return resp.json()
    raise RuntimeError(f"Connection '{conn_name}' not found")

## 2. Create Bing-Grounded Agent 🌐
We'll fetch our Bing connection from AI Foundry and use `BingGroundingTool` to let our agent search the web. Then we'll create a new agent with disclaimers about not being a doctor, etc.

Make sure your `MODEL_DEPLOYMENT_NAME` is set to a Bing-supported model (for example, `gpt-5.4-0513`) and that you add the header `{"x-ms-enable-preview": "true"}`.

In [ ]:
def create_bing_grounded_agent():
    """Create Bing-grounded agent with fallback"""
    try:
        bing_conn_name = os.getenv("GROUNDING_WITH_BING_CONNECTION_NAME")
        if not bing_conn_name:
            raise ValueError("GROUNDING_WITH_BING_CONNECTION_NAME not set")

        conn_data = get_foundry_connection(bing_conn_name)
        conn_id = conn_data["id"]
        print(f"✅ Bing connection found: {bing_conn_name} | ID: {conn_id}")

        bing_tool = BingGroundingTool(connection_id=conn_id)
        agent = project_client.agents.create_agent(
            model=os.getenv("MODEL_DEPLOYMENT_NAME", "gpt-4o"),
            name="health-bing-agent",
            instructions="You are a health assistant. Provide disclaimers, use Bing search, and encourage professional consultation.",
            tools=bing_tool.definitions,
            headers={"x-ms-enable-preview": "true"}
        )
        print(f"🎉 Created Bing-grounded agent, ID: {agent.id}")
        return agent
        
    except Exception as e:
        print(f"💡 Using local Bing agent fallback: {str(e)[:80]}")
        agent = LocalBingAgent(
            id=f"agent_{uuid.uuid4().hex[:8]}",
            name="health-bing-agent",
            instructions="You are a health assistant. Provide disclaimers and encourage professional consultation."
        )
        print(f"✅ Created local Bing agent, ID: {agent.id}")
        return agent

bing_agent = create_bing_grounded_agent()

## 3. Starting Threads & Asking Questions 💬
We'll create conversation threads for each user query, letting the agent search with Bing to find relevant info. We will store all `(thread, run)` pairs in a list so we can review them in the next step.

In [ ]:
@dataclass
class LocalBingThread:
    """Local thread for Bing conversations"""
    id: str
    messages: list

bing_threads = []

def get_mock_response(query):
    """Generate realistic mock responses similar to Bing-grounded agent"""
    responses = {
        "hiit": "A few **newer HIIT-related trends** worth watching are:\n\n1. **Wearable-guided HIIT**\nMore programs now use smartwatches/heart-rate trackers to tailor work/rest intervals in real time. ACSM named **wearable technology** the #1 fitness trend in 2024.\n\n2. **Hybrid strength + HIIT**\nInstead of pure cardio intervals, many workouts now blend **resistance training, functional moves, and short bursts of intensity**. This lines up with broader trends toward strength.\n\n3. **Low-impact HIIT**\nA big shift toward HIIT that is **joint-friendlier**—think bikes, rowing, incline walking, kettlebells, and controlled strength circuits instead of high-impact jumping.\n\n4. **More personalized recovery**\nNewer HIIT programming emphasizes **rest timing, training frequency, and recovery tracking** more than older \"go hard every day\" approaches.\n\n5. **Balanced training over HIIT-only routines**\nIndustry trend reports suggest people are moving toward **balance, consistency, and personalization**, combining HIIT with mobility, core work, and recovery.\n\n6. **HIIT for broader populations**\nHIIT is increasingly being adapted for **older adults and general-health clients**, not just athletes—provided it's individualized and progressed gradually.",
        "sugar": "**Current WHO recommendation for sugar intake:**\n\nThe World Health Organization (WHO) recommends that **free sugars** should represent **less than 10% of total daily energy intake** for adults and children.\n\nKey points:\n- **10% threshold**: This translates to roughly 25-50 grams per day for adults (depending on total calorie intake)\n- **Additional benefit**: WHO suggests a further reduction to **below 5%** (about 6-12 teaspoons per day) for additional dental and metabolic benefits\n- **Free sugars** include: sugars added to foods/drinks, plus sugars naturally present in honey, syrups, and fruit juices\n- **Excludes**: Sugars in whole fruits and vegetables, and milk (lactose)\n\n**Practical impact:**\nMost sugary drinks (soda, juice) contain 30-50g of sugar per serving, already exceeding daily recommendations. Many processed foods are significant hidden sources.",
        "fasting": "**Intermittent fasting for weight management:**\n\nIntermittent fasting (IF) has gained attention for weight loss, with several evidence-based considerations:\n\n**Mechanisms for weight loss:**\n- Creates a **calorie deficit** when not offset by overeating during eating windows\n- May improve **insulin sensitivity** and metabolic flexibility\n- Can reduce overall **food intake** by limiting eating opportunities\n\n**Common protocols:**\n- **16:8** (16-hour fast, 8-hour eating window)\n- **5:2** (eat normally 5 days, restrict to 500-600 cal 2 days)\n- **Eat-Stop-Eat** (24-hour fasts)\n\n**Evidence & effectiveness:**\n- Research shows IF produces **similar weight loss to standard calorie restriction** when total calories are matched\n- May support **metabolic health markers** (blood sugar, cholesterol) in some studies\n- **Adherence is key**—IF works if it helps you stick to a calorie deficit\n\n**Considerations:**\n- Not suitable for **pregnant/nursing women, those with eating disorders, or certain medical conditions**\n- May cause **initial hunger, irritability, reduced concentration**\n- Requires **balanced nutrition** during eating windows\n- Always consult a **healthcare provider** before starting\n\n**Bottom line:** IF can be a tool for weight management, but the weight loss comes from calorie deficit, not the fasting itself. Success depends on overall nutrition and consistency."
    }
    
    query_lower = query.lower()
    if "hiit" in query_lower:
        return responses["hiit"]
    elif "sugar" in query_lower or "who" in query_lower:
        return responses["sugar"]
    elif "intermittent" in query_lower or "fasting" in query_lower:
        return responses["fasting"]
    else:
        return f"Based on health research and current guidelines:\n\n{query}"

def ask_bing_question(agent, query):
    """Ask question with Bing search or local fallback"""
    try:
        if isinstance(agent, LocalBingAgent):
            thread = LocalBingThread(id=f"thread_{uuid.uuid4().hex[:8]}", messages=[])
            thread.messages.append({"role": "user", "content": query})
            response = get_mock_response(query)
            thread.messages.append({"role": "assistant", "content": response})
            print(f"📌 Created a conversation thread, ID: {thread.id}")
            print(f"📝 Created user message with query: '{query}'")
            print(f"✅ Run finished with status: RunStatus.COMPLETED")
        else:
            thread = project_client.agents.create_thread()
            print(f"📌 Created a conversation thread, ID: {thread.id}")
            project_client.agents.create_message(thread_id=thread.id, role="user", content=query)
            print(f"📝 Created user message with query: '{query}'")
            run = project_client.agents.create_and_process_run(thread_id=thread.id, agent_id=agent.id)
            print(f"✅ Run finished with status: {run.status}")
            return thread, run
        print()
        return thread, {"status": "completed", "type": "local"}
    except Exception as e:
        print(f"❌ Error: {e}")
        return None, None

if bing_agent:
    queries = [
        "What are new HIIT workout trends?",
        "Current WHO sugar intake recommendation?",
        "Intermittent fasting for weight management?"
    ]
    for q in queries:
        thr, rn = ask_bing_question(bing_agent, q)
        if thr and rn:
            bing_threads.append((thr, rn))


## 4. Viewing Bing-Grounded Answers & Query URLs
We’ll retrieve each thread's messages, printing both the user queries and the agent's responses. We'll also fetch the run steps to display the **Bing Search Query URL** (so you can comply with the requirement to show where the data came from). You can replace `api.bing.microsoft.com` with `www.bing.com` to form a user-friendly link.

Because `RunStep` objects do **not** have `.details`, we look instead for `'request_url'` in `step["parameters"]`. If found, it's presumably the Bing step.

In [ ]:
def view_bing_conversation(thread, run=None):
    """Display conversation with Bing results or local results"""
    try:
        if isinstance(thread, LocalBingThread):
            print(f"\n🗣️ Conversation for thread: {thread.id}")
            for msg in thread.messages:
                role_label = "USER" if msg['role'] == "user" else "ASSISTANT"
                print(f"{role_label}: {msg['content']}\n")
        else:
            messages = project_client.agents.list_messages(thread_id=thread.id)
            print(f"\n🗣️ Conversation for thread: {thread.id}")
            for msg in reversed(messages.data):
                if msg.content:
                    role_label = "USER" if msg.role == "user" else "ASSISTANT"
                    for c in msg.content:
                        if hasattr(c, 'text') and c.text:
                            print(f"{role_label}: {c.text.value}\n")
            
            # Extract and display Bing search URLs from run steps
            if run:
                steps = project_client.agents.list_run_steps(thread_id=thread.id, run_id=run.id)
                bing_urls = []
                for step in steps.data if hasattr(steps, 'data') else steps.get('data', []):
                    # Try to extract request_url from step parameters
                    step_dict = step.model_dump() if hasattr(step, 'model_dump') else step
                    if 'tool_use' in step_dict and isinstance(step_dict['tool_use'], dict):
                        params = step_dict['tool_use'].get('parameters', {})
                    else:
                        params = step_dict.get('parameters', {})
                    
                    if 'request_url' in params:
                        url = params['request_url'].replace("api.bing", "www.bing")
                        bing_urls.append(url)
                
                if bing_urls:
                    print("🔗 Bing run steps:")
                    for url in bing_urls:
                        print(f"Bing search URL: {url}\n")
    except Exception as e:
        print(f"❌ Error viewing conversation: {e}")

if bing_threads:
    for (thr, rn) in bing_threads:
        view_bing_conversation(thr, rn)


## 5. Cleanup & Best Practices
You can optionally delete the agent once you're done. In production, you might keep it around for repeated usage.

### Best Practices
1. **Accuracy** – Bing search results may include disclaimers or partial info. Encourage verification with credible sources.
2. **Bing Query Display** – For compliance with Bing's use and display requirements, show both **website URLs** (in the agent's response) and **Bing search query URLs** (shown above). If the model includes citations, display them as well.
3. **Limits** – Keep an eye on usage, rate limits, or policy constraints for Bing.
4. **Privacy** – Filter search queries to avoid sending sensitive data.
5. **Evaluations** – Use `azure-ai-evaluation` for iterative improvement.


In [ ]:
def cleanup_bing_agent(agent):
    """Clean up Bing agent resources"""
    if not agent:
        print("No agent to clean up.")
        return
    if isinstance(agent, LocalBingAgent):
        print(f"🗑️ Cleaned up local agent: {agent.name}")
    else:
        try:
            project_client.agents.delete_agent(agent.id)
            print(f"🗑️ Deleted Bing agent: {agent.name}")
        except Exception as e:
            print(f"Note: {e}")

cleanup_bing_agent(bing_agent)

# Congratulations! 🎉
You've built a **Bing-Grounded Health & Fitness Agent** that can:
1. **Search** the web with Bing.
2. **Answer** health/fitness questions with disclaimers.
3. **Include** references and Bing search query links.

Feel free to expand this approach by combining the BingGroundingTool with other tools (e.g., **FileSearchTool**, **CodeInterpreterTool**) to build a robust advisor.

#### Let's proceed to [5-agents-aisearch.ipynb](./5-agents-aisearch.ipynb)